In [1]:
from __future__ import division
import math
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets

from ipywidgets import interact, interactive, fixed, interact_manual, Label, Layout

colors = [
    "rgb( 31,120,180)",
    "rgb( 51,160, 44)",
    "rgb(227, 26, 28)",
    "rgb(255,128,  0)",
    "rgb(106, 61,154)",
    "rgb(177, 89, 40)",
    "rgb(224, 96,224)",
    "rgb(224,224, 96)",
    "rgb( 96,224,224)",
    "rgb(160,160,160)"
]
n_colors = len(colors)

In [2]:
# A) Initialisation des variables  #####################################################################

# 1. Constantes physiques (NIST)
h    = 6.626068e-34;  # [J.S]      Planck constant
heV  = 4.135667e-15;  # [eV.s]     Planck constant
hb   = h/(2*math.pi);      # [J.s]      h barre
hbeV = heV/(2*math.pi);    # [eV.s]     h barre
me   = 9.109381e-31;  # [kg]       electron mass
q    = 1.6021764e-19; # [C]        electron charge
c    = 299792458;     # [m.s^(-1)] speed of light in vacuum

# 2. Propriété de la particule étudiée
m = me;           # [kg] masse (me = masse de l'électron)

# 3. Propriétés de l'espace étudié (puits, maille,...)
xMIN = -1e-8 # [m] extension maximale vers la gauche
xMAX = -xMIN # [m] extension maximale vers la droite
xNum = 1001   #  [-] nombre de valeurs discrétisées de x (résolution) doit être impair
    

# 4. Niveaux énergétiques et fonctions d'onde
vectEn   = np.arange(0,10,1) # nombre niveaux énergétiques sélectionnés pour le plot
vectPSIn = np.arange(0,5,1)  # nombre fonctions d'ondes sélectionnées pour le plot

# 5 Définitions de constantes internes
dx    = (xMAX-xMIN)/(xNum-1) # [m]
xArr  = np.linspace(xMIN, xMAX, xNum-1)
xOnes = [1.0] * (xNum-1)


In [3]:
# B) Définition du potentiel ####################################################################################

def Potentiel(idxPot, nPuits, Profondeur, Largeur, Distance, fig):
    """
    Calcule et affiche un potentiel en fonction du type de potentiel (rectangulaire ou coulombien).

    Paramètres :
    ------------
    idxPot : int
        Indice pour choisir le type de potentiel (1 pour rectangulaire, autre pour coulombien).
    nPuits : int
        Nombre de puits de potentiel.
    Profondeur : float
        Profondeur des puits de potentiel (en eV).
    Largeur : float
        Largeur des puits de potentiel (en nm).
    Distance : float
        Distance entre les puits de potentiel (en nm).
    fig : objet plotly
        Figure sur laquelle le potentiel sera tracé.

    Retourne :
    -----------
    Pot : list
        Liste des valeurs du potentiel calculé pour chaque point de l'axe x.
    """

    # Définition du potentiel
    # Cas du potentiel rectangulaire
    if idxPot == 1:
        # Conversion des unités : Profondeur en eV, Largeur et Distance en mètres
        Profondeur = Profondeur * q  # q est la charge élémentaire (1.602e-19 C)
        Largeur = Largeur * 1e-9  # Conversion de nm en m
        Distance = Distance * 1e-9  # Conversion de nm en m

        # Initialisation de la liste des valeurs du potentiel
        Pot = []

        # Parcourir chaque point de l'axe x
        for ii, xx in enumerate(xArr):
            inWell = False  # Variable pour vérifier si le point xx est dans un puits

            # Cas où le nombre de puits est pair
            if nPuits % 2 == 0:
                # Parcourir chaque puits
                for ip in range(-int(nPuits/2), int(nPuits/2)):
                    # Vérifier si xx est dans le puits ip
                    if xx >= (Distance/2 + ip*(Largeur+Distance)) and xx <= (Distance/2 + Largeur + ip*(Largeur+Distance)):
                        inWell = True

            # Cas où le nombre de puits est impair
            else:
                # Parcourir chaque puits
                for ip in range(-int((nPuits-1)/2), int((nPuits+1)/2)):
                    # Vérifier si xx est dans le puits ip
                    if xx >= (-Largeur/2 + ip*(Largeur+Distance)) and xx <= (Largeur/2 + ip*(Largeur+Distance)):
                        inWell = True

            # Si xx est dans un puits, le potentiel est égal à -Profondeur, sinon 0
            if inWell:
                Pot.append(-Profondeur)
            else:
                Pot.append(0)

    # Cas du potentiel quartique
    elif idxPot == 4:
        # Conversion des unités : Profondeur en eV, Largeur et Distance en mètres
        Profondeur = Profondeur * q
        Largeur = Largeur * 1e-9
        Distance = Distance * 1e-9

        # Initialisation de la liste des valeurs du potentiel
        Pot = []

        # Parcourir chaque point de l'axe x
        for ii, xx in enumerate(xArr):
            potval = 0  # Valeur du potentiel pour le point xx

            # Un unique puits quartique centré en 0
            if nPuits == 1:
                potval = Profondeur * (xx/Largeur)**4

            else:
                raise NotImplementedError("Le potentiel quartique n'est implémenté que pour un seul puits (nPuits=1).")

            # Limiter la valeur du potentiel à -Profondeur (pour éviter des valeurs trop basses)
            Pot.append(max(potval, -Profondeur))

    # Cas du potentiel coulombien
    else:
        # Conversion des unités : Profondeur en eV, Largeur et Distance en mètres
        Profondeur = Profondeur * q
        Largeur = Largeur * 1e-9
        Distance = Distance * 1e-9

        # Initialisation de la liste des valeurs du potentiel
        Pot = []

        # Parcourir chaque point de l'axe x
        for ii, xx in enumerate(xArr):
            potval = 0  # Valeur du potentiel pour le point xx

            # Cas où le nombre de puits est pair
            if nPuits % 2 == 0:
                # Parcourir chaque puits
                for ip in range(-int(nPuits/2), int(nPuits/2)):
                    # Calculer la contribution de chaque puits au potentiel total
                    potval += -q/abs((xx-ip*(Largeur+Distance)-(Largeur+Distance)/2)/(Largeur/20))

            # Cas où le nombre de puits est impair
            else:
                # Parcourir chaque puits
                for ip in range(-int((nPuits-1)/2), int((nPuits+1)/2)):
                    # Calculer la contribution de chaque puits au potentiel total
                    potval += -q/abs((xx-ip*(Largeur+Distance))/(Largeur/20))

            # Limiter la valeur du potentiel à -Profondeur (pour éviter des valeurs trop basses)
            Pot.append(max(potval, -Profondeur))

    # Affichage du potentiel sur la figure
    fig.add_trace(
        go.Scatter(
            x=xArr*10**9,  # Conversion de x en nm pour l'affichage
            y=np.array(Pot)/q,  # Normalisation par q pour l'affichage
            mode='lines',
            line=dict(color='black'),
            name='Potentiel'
        ),
        row=1, col=1
    )

    # Retourner la liste des valeurs du potentiel
    return Pot

In [8]:
# C) Calcul des états propres par différences finies ########################################################

def Schroedinger(Pot,nEn,fig):
    

    # 1. Définition des diagonales de la matrice T
    diag2 = [1.0] * int(xNum-2)
    diag3 = [1.0] * int(xNum-3)

    # Matrice de l'energie cinétique T
    matDiff = -2*np.diag(diag2,k=0) + np.diag(diag3,k=1) + np.diag(diag3,k=-1)
    matKin = -(hb**2)/(2*m) * matDiff/(dx**2)

    # Matrice de l'energie potentielle
    matPot  = np.diag(Pot[1:int(xNum-1)],k=0)

    # 2. Hamiltonien de l'equation de Schroedinger H = (a A + B ) with a = -(hb**2)/(2* m * dx**2) 
    H = matKin + matPot

    # 3. Résolution de l'équation aux valeurs propres H Psi = E Psi avec ``Psi[:,i]`` le vecteur propre normalise 
    # correspondant a la valeur propre ``E[i]``.
    E, Psi = np.linalg.eig(H)

    # Tri en ordre croissant des valeurs propres et des vecteurs d'ondes correspondants
    E_sorted = np.sort(E)
    sorted_Psi = Psi[:,np.argsort(E)]
    # Affichage des premiers niveaux energétiques
    vectEn = np.arange(0,nEn,1)
    
    for ii in vectEn:
        fig.add_trace(
            go.Scatter(
                x=xArr[1:None]*10**9, y=[E_sorted[ii]/q]*len(xArr), line=dict(color=colors[ii%n_colors]), name='E' + str(ii)
            ),
            row=1, col=1
        )
        print("E"+str(ii)+" = "+str(E_sorted[ii]/q)+" eV")

    fig.update_xaxes(title_text="Distance (nm)", range=[xMIN*10**9, xMAX*10**9], row=1, col=1)
    fig.update_yaxes(title_text="Energie (eV)", row=1, col=1)

    # Fonctions d'ondes
    # ajouter Phi_0 et Phi_n = 0
    Psi_new = np.insert(sorted_Psi,0,0,axis=1)
    Psi_new = np.insert(Psi_new,xNum-1,0,axis=1)
    # norme au carré de Psi 
    PSI2 = np.conj(Psi_new)*Psi_new
    PSIMAX = np.max(PSI2)
    #affichage des fonctions d'ondes
    vectPSIn = np.arange(0,nEn,1)
    for ii in vectPSIn:
        fig.add_trace(
            go.Scatter(
                x=xArr[1:None]*10**9, y=PSI2[:,ii+1]/PSIMAX, line=dict(color=colors[ii%n_colors]), name='|Psi'+str(ii)+'|^2'
                # x=xArr[1:None]*10**9, y=Psi_new[:,ii+1], line=dict(color=colors[ii%n_colors]), name='Psi'+str(ii)
            ),
        row=1, col=2
    )

    # fig.update_xaxes(title_text="Distance (nm)", range=[xMIN*10**9, xMAX*10**9], row=1, col=2)
    fig.update_yaxes(title_text="Densité de probabilité", row=1, col=2)
    fig.update_yaxes(title_text="Fonction d'onde", row=1, col=2)

In [9]:
def func(potentiel,profondeur,largeur,nPuits,distance,nEn):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Potentiel", "Fonctions d'onde"),
        shared_xaxes=False,
        shared_yaxes=False
    )
    pot = Potentiel(idxPot=potentiel,nPuits=nPuits,Profondeur=profondeur,Largeur=largeur,Distance=distance,fig=fig)
    Schroedinger(pot,nEn,fig)
    fig.update_layout(height=500, width=1200)
    fig.show()

In [10]:
potentiel = widgets.Select(
    options=[
        ('1. Rectangulaire',1),
        ('2. Coulombien',   2),
        ('4. Quartique',   4),
    ],
    disabled=False,
    layout=Layout(width='150px', height='150px'))

profondeur = widgets.FloatSlider(value=0.5, min=0.0001, max=2,  step=0.1, continuous_update=False, orientation='vertical')
largeur    = widgets.FloatSlider(value=2,   min=0.1,    max=10, step=0.1,   continuous_update=False, orientation='vertical')
distance   = widgets.FloatSlider(value=2,   min=0,      max=10, step=0.1,   continuous_update=False, orientation='vertical')
nPuits     = widgets.IntSlider(value=1,     min=1,      max=4,  step=1,     continuous_update=False, orientation='vertical')
nEn        = widgets.IntSlider(value=4,     min=1,      max=10, step=1,     continuous_update=False, orientation='vertical')

box1 = widgets.VBox([Label('Choix du potentiel'),potentiel])
box2 = widgets.VBox([Label('# puits'),nPuits])
box3 = widgets.VBox([Label('Profondeur (eV)'),profondeur])
box4 = widgets.VBox([Label('Largeur (nm)'),largeur])
box5 = widgets.VBox([Label('Distance (nm)'),distance])
box6 = widgets.VBox([Label('# états'),nEn])

ui = widgets.HBox([box1, box2, box3, box4, box5, box6])

out = widgets.interactive_output(
    func,{
        'potentiel':potentiel,
        'nPuits':nPuits,
        'profondeur':profondeur,
        'largeur':largeur,
        'distance':distance,
        'nEn':nEn
    }
)

display(ui,out)


Output()